In [0]:
--Testing LAG(col)

SELECT
  o_custkey,
  o_orderkey,
  o_orderdate,
  LAG(o_orderdate, 1, NULL) OVER (
    PARTITION BY o_custkey
    ORDER BY o_orderdate, o_orderkey
  ) AS prev_order_date
FROM samples.tpch.orders
WHERE o_custkey = 1
ORDER BY o_orderdate, o_orderkey;

In [0]:
--Expanded LAG(col) with DateDiff

SELECT
  o_custkey,
  o_orderkey,
  o_orderdate,
  LAG(o_orderdate) OVER (
    PARTITION BY o_custkey
    ORDER BY o_orderdate, o_orderkey
  ) AS prev_order_date,
  DATEDIFF(o_orderdate, prev_order_date) AS days_since_prev
FROM samples.tpch.orders
ORDER BY o_orderdate, o_orderkey
LIMIT 150;

In [0]:
SELECT
  o_custkey,
  o_orderkey,
  o_orderdate,
  ROW_NUMBER() OVER (
    PARTITION BY o_custkey
    ORDER BY o_orderdate, o_orderkey
  ) AS rn
FROM samples.tpch.orders
ORDER BY o_orderdate, o_orderkey
LIMIT 50;

In [0]:
--Find the 10 customers with the longest gap between two consecutive orders. Return o_custkey, the gap in days, and the two dates.

WITH datediff AS (
    SELECT
        o_custkey,
        DATEDIFF(o_orderdate, LAG(o_orderdate) OVER (
            PARTITION BY o_custkey
            ORDER BY o_orderdate, o_orderkey
        )) AS days_since_prev
    FROM samples.tpch.orders
)
SELECT *
FROM (
    SELECT
        o_custkey,
        days_since_prev,
        ROW_NUMBER() OVER (
            ORDER BY days_since_prev DESC
        ) AS rn
    FROM datediff
)
WHERE rn <= 10
ORDER BY days_since_prev DESC



In [0]:
--Find the 10 customers with the longest gap between two consecutive orders. Return o_custkey, the gap in days, and the two dates.

WITH datediff AS (
    SELECT
        o_custkey,
        DATEDIFF(o_orderdate, LAG(o_orderdate) OVER (
            PARTITION BY o_custkey
            ORDER BY o_orderdate, o_orderkey
        )) AS days_since_prev
    FROM samples.tpch.orders
)
SELECT o_custkey, days_since_prev from datediff
ORDER BY days_since_prev DESC
LIMIT 10;




In [0]:
-- LEAD

SELECT
  o_custkey,
  o_orderkey,
  o_orderdate,
  LEAD(o_orderdate, 1, NULL) OVER (
    PARTITION BY o_custkey
    ORDER BY o_orderdate, o_orderkey
  ) AS prev_order_date
FROM samples.tpch.orders
WHERE o_custkey = 1
ORDER BY o_orderdate, o_orderkey;

In [0]:
-- 3a: test fan-out bug:
SELECT
  n.n_name,
  SUM(o.o_totalprice) as total_revenue
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation AS n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.lineitem AS l ON o.o_orderkey = l.l_orderkey
GROUP BY n.n_name
ORDER BY total_revenue DESC
LIMIT 5;

In [0]:
--solution for fan-out bug

SELECT
  n.n_name,
  SUM(l_extendedprice * (1 - l_discount)) as total_revenue
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation AS n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.lineitem AS l ON o.o_orderkey = l.l_orderkey
GROUP BY n.n_name
ORDER BY total_revenue DESC
LIMIT 5;

In [0]:
--Fix B: keep summing o_totalprice, but pre-aggregate lineitem to one row per order in a CTE (for example, the number of lines per order) before joining. The revenue should then match Day 1 exactly.


WITH lineitem_agg AS (
    SELECT l_orderkey, count(*) as num_lines
    FROM samples.tpch.lineitem
    GROUP BY l_orderkey
)
SELECT
  n.n_name,
  SUM(o.o_totalprice) as total_revenue
FROM samples.tpch.orders AS o
JOIN samples.tpch.customer AS c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation AS n ON c.c_nationkey = n.n_nationkey
JOIN lineitem_agg AS l ON o.o_orderkey = l.l_orderkey
GROUP BY n.n_name
ORDER BY total_revenue DESC
LIMIT 5;

In [0]:
-- 3c. Find customers with no orders.

SELECT COUNT(*) AS custwoorders
FROM (
  SELECT c.c_custkey, c.c_name
  FROM samples.tpch.customer AS c
  LEFT JOIN samples.tpch.orders AS o ON c.c_custkey = o.o_custkey
  WHERE o.o_orderkey IS NULL
);

In [0]:
SELECT COUNT(*) AS custwoorders
FROM (
  SELECT c.c_custkey, c.c_name
  FROM samples.tpch.customer AS c
  LEFT ANTI JOIN samples.tpch.orders AS o ON c.c_custkey = o.o_custkey
);


In [0]:
SELECT ((SELECT COUNT(*) FROM samples.tpch.customer)-(SELECT COUNT(DISTINCT o_custkey) FROM samples.tpch.orders )) AS custwoorders;

In [0]:
-- Intent: all customers, with their 1998 orders if they have any
SELECT COUNT(*) AS 1998orders
FROM (SELECT c.c_custkey, o.o_orderkey
FROM samples.tpch.customer c
LEFT JOIN samples.tpch.orders o ON c.c_custkey = o.o_custkey
WHERE YEAR(o.o_orderdate) = 1998);   -- ❌ drops customers with no match

In [0]:

SELECT COUNT(*) AS 1998orders
FROM (SELECT c.c_custkey, o.o_orderkey
FROM samples.tpch.customer c
LEFT JOIN samples.tpch.orders o
  ON c.c_custkey = o.o_custkey
 AND YEAR(o.o_orderdate) = 1998 );    -- ✅ filters orders, keeps all customers